In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 KB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 19.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 91.7 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 5.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 KB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 25.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.9 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 KB 74.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 42.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 KB 70.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 196.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 67.2 

In [2]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [3]:
# Find the number of CPUs/cores on the machine
!nproc --all

256


In [4]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 256


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

In [5]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA L40 (UUID: GPU-fcd7d268-efb6-a1bf-5dc8-5526fd331413)


In [6]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/venv/main/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/venv/main/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/venv/main/lib/python3.10/site-packages/ipykernel/kernelapp.p

True
NVIDIA L40
Free GPU memory: 45055.375 out of: 45487.75


In [7]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Translate

In [8]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
!onmt_translate -model models/model.fren_step_10000.pt -src en-zh.en-filtered.en.subword.test -output zh.base.translated -gpu 0 -min_length 1




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_translate", line 5, in <module>
    from onmt.bin.translate import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/ven

In [14]:
%pip install "numpy<2"


Note: you may need to restart the kernel to use updated packages.


In [24]:
# Check the first 5 lines of the translation file
!head -n 30 zh.base.translated

▁我 还没 意识到 , ▁她 就已经 从 停车场 滑 过 , ▁在 车 里 和 车 之间 , ▁还有 我 身 后 的人 , ▁ 带着 那种 ▁“ 我 来 。” 意大利 手 势 跟着 。
▁如果我 把 乌干达 分解 开 , 在 乌干达 有很大的 不同 。
▁我 看着 这张照片 , ▁他 很 感兴趣 的是 要 用 那个 按钮 , ▁但是 看起来 他 并不 感兴趣 的是 要 过 马路 。
▁我不知道 , ▁ 这个项目 有很多 可能性 , ▁我 鼓励 你们 所有人 , ▁ 记录 下 你 生活的 一个小 片段 , ▁所以你 永远 忘 不了 那天 , 你 真的 活着 。
▁它 来自 一个叫 p a ch y al os 的 染色体 。
▁我 回到 希 思 罗 的 身边 。 我妈妈 在那里 ; ▁我 的 哥哥 在那里 ; 我的 孙 子 在那儿 。 ▁ 那儿 有个 小 国 旗 。 ▁就是 关于 这个 的 。 我 回去 和 我妈妈 住在 一起 。
▁这不是 一个 修 辞 性 的问题 。
▁ 好消息 是 这些 领袖 大部分 都 已经 习惯 了 , ▁他们 被 三 代 人 所 取代 。
▁ 有一天 , 我在 S k y p e 通 话 , ▁他 正在 新 建 的 佛罗里达 大学 , ▁在 公共 卫生 领域 , ▁他 很 自豪 地 告诉我 , 他 可以 ▁从 美国 公众 那里 筹 集 足够 的 资金 , ▁ 以此 为 基础 。 ▁我想 带 你们 回到 ▁H an y 。
▁我想 快速 地 展示 一下 。
▁CA : ▁但是 弦 理论 家 , 我 理解 它 , ▁ 解释 了 电子 的 多少 弦 来 震动 -- ▁我知道 你 不喜欢 弦 理论 -- 在 它 内部 震动 。
▁ 我最喜欢的 一个 是 : ▁ 左 宗 棠 鸡 , ▁ 顺便 说 一句 , 在美国 海 军 学院 ▁ 叫做 阿 里 米 · 特 尔 鸡 。
▁根据 我们的 标准 , 有些人 将 ▁ 视为 原始 的 。
▁当 所有这些 方法 , ▁我们 被 鼓励 去 看待 别人 , ▁ 带着 敌 意 、 恐惧 和 怀疑 。
▁所以我 说 , ▁“ 哦 , 这是个 额外 的 成分 , ▁你知道 , 你需要 来自 母亲 和 青蛙 。” ▁她说 ,“ 哦 , 人类 也 一样 吗 ?”
▁由 一个 社会 科学家 来 捐 献 , ▁他们说 他们 不知

In [11]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.base.translated

Done desubwording! Output: zh.base.translated.desubword


In [2]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered.zh.subword.test

Done desubwording! Output: en-zh.en-filtered.en.subword.test.desubword
Done desubwording! Output: en-zh.zh-filtered.zh.subword.test.desubword


In [8]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.base.translated.desubword

我还没意识到, 她就已经从停车场滑过, 在车里和车之间, 还有我身后的人, 带着那种 “我来。”意大利手势跟着。
如果我把乌干达分解开,在乌干达有很大的不同。
我看着这张照片, 他很感兴趣的是要用那个按钮, 但是看起来他并不感兴趣的是要过马路。
我不知道, 这个项目有很多可能性, 我鼓励你们所有人, 记录下你生活的一个小片段, 所以你永远忘不了那天,你真的活着。
它来自一个叫pachyalos的染色体。
我回到希思罗的身边。我妈妈在那里; 我的哥哥在那里;我的孙子在那儿。 那儿有个小国旗。 就是关于这个的。我回去和我妈妈住在一起。
这不是一个修辞性的问题。
好消息是这些领袖大部分都已经习惯了, 他们被三代人所取代。
有一天,我在Skype通话, 他正在新建的佛罗里达大学, 在公共卫生领域, 他很自豪地告诉我,他可以 从美国公众那里筹集足够的资金, 以此为基础。 我想带你们回到 Hany。
我想快速地展示一下。
CA: 但是弦理论家,我理解它, 解释了电子的多少弦来震动-- 我知道你不喜欢弦理论--在它内部震动。
我最喜欢的一个是: 左宗棠鸡, 顺便说一句,在美国海军学院 叫做阿里米·特尔鸡。
根据我们的标准,有些人将 视为原始的。
当所有这些方法, 我们被鼓励去看待别人, 带着敌意、恐惧和怀疑。
所以我说, “哦,这是个额外的成分, 你知道,你需要来自母亲和青蛙。” 她说,“哦,人类也一样吗?”
由一个社会科学家来捐献, 他们说他们不知道用什么, 如何使用,所以——抱歉。
你上网搜索, 你会看到类似Codemery、Coodorjo 还有一些网站,例如“少女”、“黑人女孩子”之类的网站。
我可以用她跟着我笑。
那一天到来之前, 我代表我的父亲, 曾试着用一把老枪射杀纳粹。
所以我得到了一个警察 Abed继续在他小镇的某个地方生活着 而我现在正带着装满黄色的玫瑰 ⁇ 坐在后座, 突然,花看上去很荒谬。
没有时间详细介绍, 但是也包括 树种护士、 农业耕种的方法, 尤其是现在退化的, 几乎在这些山里有沙漠中的土地。
我们都需要觉得自己重要,特别,独特。
对我来说,作为一个公共卫生学教授, 这些国家发展得如此迅速。
所以我们一直提出的问题是, 全世界到底用了多少地方来种植粮食, 具体在哪里,我们怎样才能改变它们的未来, 又意味着什么?
一些战斗过高压政府。
当她的情况变得

## Evaluation

In [21]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-03 01:26:46--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py.2’

compute-bleu.py.2   100%[===================>]     957  --.-KB/s    in 0s      

2025-04-03 01:26:46 (103 MB/s) - ‘compute-bleu.py.2’ saved [957/957]



In [22]:
# Install sacrebleu
!pip3 install sacrebleu

In [9]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

Reference 1st sentence: 我还没回过神儿,她已轻松穿行于停车场的汽车中, 身后的人们看我的眼神, 充满了惊羡。哇啊——哇啊——
MTed 1st sentence: 我还没意识到, 她就已经从停车场滑过, 在车里和车之间, 还有我身后的人, 带着那种 “我来。”意大利手势跟着。
BLEU:  2.410088986549132
